# Feature Engineering v6: точечное улучшение v5

## Ключевые отличия от v5:
1. Винзоризация (клиппинг) таргета при обучении на отметке 4000 по 99.9 перцентилю
2. Добавление флага для редких покупателей
3. После 120 дней конверсия вероятности покупки почти перестает уменьшаться, добавлен отдельный флаг для этого

In [ ]:
import polars as pl
import numpy as np
from pathlib import Path
from datetime import date, timedelta
from typing import Optional
import time

DATA_DIR = Path("../data/raw")
FEATURES_DIR = Path("../data/processed/features_v6")
FEATURES_DIR.mkdir(parents=True, exist_ok=True)

N_FOLDS = 5
STRIDE_DAYS = 14
MIN_HISTORY = 180
BATCH_SIZE = 50_000
TARGET_COL = "gmv"

VALUE_COLS = [
    "gmv", "searches", "to_cart", "to_ord",
    "search_to_cart", "search_to_ord",
    "cat_to_cart", "cat_to_ord",
    "gmv_search", "gmv_cat",
    "search", "cat",
]

WINDOWS = [
    ("3d",   2,   0),
    ("7d",   6,   0),
    ("14d",  13,  0),
    ("30d",  29,  0),
    ("60d",  59,  0),
    ("90d",  89,  0),
    ("180d", 179, 0),
    ("270d", 269, 0),
    ("365d", 364, 0),
]

print(f"N_FOLDS={N_FOLDS}, stride={STRIDE_DAYS}, min_history={MIN_HISTORY}")
print(f"VALUE_COLS: {len(VALUE_COLS)}, WINDOWS: {len(WINDOWS)}")

N_FOLDS=5, stride=14, min_history=180
VALUE_COLS: 12, WINDOWS: 9


In [10]:
def generate_cv_anchor_dates(
    data: pl.DataFrame,
    prediction_horizon_days: int = 30,
    stride_days: int = STRIDE_DAYS,
    min_history_days: int = MIN_HISTORY,
    n_folds: Optional[int] = None,
) -> list[date]:
    min_date = data["event_date"].min()
    max_date = data["event_date"].max()
    latest_anchor = max_date - timedelta(days=prediction_horizon_days)
    earliest_anchor = min_date + timedelta(days=min_history_days - 1)
    n_steps = (latest_anchor - earliest_anchor).days // stride_days
    all_anchors = sorted([latest_anchor - timedelta(days=i * stride_days) for i in range(n_steps + 1)])
    if n_folds:
        return all_anchors[-n_folds:]
    return all_anchors

In [11]:
def _build_feature_exprs(anchor_val: date, windows: list, value_cols: list) -> list[pl.Expr]:
    exprs = []
    for w_name, start_off, end_off in windows:
        w_start = anchor_val - timedelta(days=start_off)
        w_end = anchor_val - timedelta(days=end_off)
        mask = pl.col("event_date").is_between(w_start, w_end)
        for col in value_cols:
            exprs.append(pl.when(mask).then(pl.col(col)).otherwise(0.0).sum().alias(f"{col}_sum_{w_name}"))
            exprs.append(pl.when(mask).then(pl.col(col)).otherwise(None).max().alias(f"{col}_max_{w_name}"))
            exprs.append(pl.when(mask).then(pl.col(col)).otherwise(None).mean().alias(f"{col}_mean_{w_name}"))
        exprs.append(pl.when(mask & (pl.col("searches") > 0)).then(1).otherwise(0).sum().alias(f"freq_search_{w_name}"))
        exprs.append(pl.when(mask & (pl.col("to_ord") > 0)).then(1).otherwise(0).sum().alias(f"freq_ord_{w_name}"))
        exprs.append(pl.when(mask & (pl.col("to_cart") > 0)).then(1).otherwise(0).sum().alias(f"freq_cart_{w_name}"))
        exprs.append(pl.when(mask & (pl.col("cat") > 0)).then(1).otherwise(0).sum().alias(f"freq_cat_{w_name}"))
        exprs.append(pl.when(mask & ((pl.col("searches") > 0) | (pl.col("cat") > 0))).then(1).otherwise(0).sum().alias(f"active_days_{w_name}"))

    exprs.append((pl.lit(anchor_val) - pl.col("event_date").filter(pl.col("to_ord") > 0).max()).dt.total_days().fill_null(999.0).alias("recency_ord_days"))
    exprs.append((pl.lit(anchor_val) - pl.col("event_date").filter(pl.col("searches") > 0).max()).dt.total_days().fill_null(999.0).alias("recency_search_days"))
    exprs.append((pl.lit(anchor_val) - pl.col("event_date").filter(pl.col("to_cart") > 0).max()).dt.total_days().fill_null(999.0).alias("recency_cart_days"))
    exprs.append((pl.lit(anchor_val) - pl.col("event_date").filter(pl.col("cat") > 0).max()).dt.total_days().fill_null(999.0).alias("recency_cat_days"))
    exprs.append((pl.lit(anchor_val) - pl.col("event_date").min()).dt.total_days().fill_null(999.0).alias("user_age_days"))
    return exprs

In [12]:
def _build_interpurchase_exprs(anchor_val: date) -> list[pl.Expr]:
    exprs = []
    
    exprs.append(pl.col("event_date").filter(pl.col("to_ord") > 0).n_unique().alias("n_order_days"))
    
    exprs.append((pl.col("event_date").filter(pl.col("to_ord") > 0).n_unique() == 1).cast(pl.Float64).alias("is_single_buyer"))
    
    ord_gaps = pl.col("event_date").filter(pl.col("to_ord") > 0).sort().diff().dt.total_days().drop_nulls()
    exprs.append(ord_gaps.mean().fill_null(999.0).cast(pl.Float64).alias("mean_order_gap"))
    exprs.append(ord_gaps.std().fill_null(999.0).cast(pl.Float64).alias("std_order_gap"))
    exprs.append(ord_gaps.min().fill_null(999.0).cast(pl.Float64).alias("min_order_gap"))
    exprs.append(ord_gaps.max().fill_null(999.0).cast(pl.Float64).alias("max_order_gap"))
    exprs.append(ord_gaps.median().fill_null(999.0).cast(pl.Float64).alias("median_order_gap"))
    
    exprs.append(pl.col("event_date").filter(pl.col("to_cart") > 0).n_unique().alias("n_cart_days"))
    cart_gaps = pl.col("event_date").filter(pl.col("to_cart") > 0).sort().diff().dt.total_days().drop_nulls()
    exprs.append(cart_gaps.mean().fill_null(999.0).cast(pl.Float64).alias("mean_cart_gap"))
    exprs.append(cart_gaps.std().fill_null(999.0).cast(pl.Float64).alias("std_cart_gap"))
    
    exprs.append(pl.col("event_date").filter((pl.col("searches") > 0) | (pl.col("cat") > 0)).n_unique().alias("n_active_days_total"))
    act_gaps = pl.col("event_date").filter((pl.col("searches") > 0) | (pl.col("cat") > 0)).sort().diff().dt.total_days().drop_nulls()
    exprs.append(act_gaps.mean().fill_null(999.0).cast(pl.Float64).alias("mean_active_gap"))
    exprs.append(act_gaps.std().fill_null(999.0).cast(pl.Float64).alias("std_active_gap"))
    
    exprs.append((pl.col("gmv").filter(pl.col("to_ord") > 0).mean()).fill_null(0.0).alias("avg_gmv_per_order_day"))
    exprs.append((pl.col("gmv").filter(pl.col("to_ord") > 0).std()).fill_null(0.0).alias("std_gmv_per_order_day"))
    
    return exprs


def _build_lifecycle_exprs(anchor_val: date) -> list[pl.Expr]:
    exprs = []
    
    periods = [("w1", 0, 6), ("w2", 7, 13), ("w3", 14, 27), ("w4", 28, 55)]
    for period, start, end in periods:
        w_start = anchor_val - timedelta(days=end)
        w_end = anchor_val - timedelta(days=start)
        mask = pl.col("event_date").is_between(w_start, w_end)
        exprs.append(pl.when(mask).then(pl.col("gmv")).otherwise(0.0).sum().alias(f"gmv_period_{period}"))
        exprs.append(pl.when(mask).then(pl.col("to_ord")).otherwise(0.0).sum().alias(f"ord_period_{period}"))
        exprs.append(
            pl.when(mask & ((pl.col("searches") > 0) | (pl.col("cat") > 0)))
            .then(1).otherwise(0).sum().alias(f"active_period_{period}")
        )
    
    for w_name, days_back in [("30d", 29), ("90d", 89), ("180d", 179)]:
        w_start = anchor_val - timedelta(days=days_back)
        mask = pl.col("event_date").is_between(w_start, anchor_val)
        s_sum = pl.when(mask).then(pl.col("gmv_search")).otherwise(0.0).sum()
        g_sum = pl.when(mask).then(pl.col("gmv")).otherwise(0.0).sum()
        exprs.append((s_sum / (g_sum + 1.0)).alias(f"search_share_{w_name}"))
    
    return exprs

In [13]:
def _build_secondary_exprs() -> list[pl.Expr]:
    eps = 1.0
    exprs = []
    
    exprs.extend([
        (pl.col("gmv_sum_7d") / (pl.col("gmv_sum_30d") + eps)).alias("trend_gmv_7d_30d"),
        (pl.col("searches_sum_7d") / (pl.col("searches_sum_30d") + eps)).alias("trend_search_7d_30d"),
        (pl.col("gmv_sum_30d") / (pl.col("gmv_sum_90d") + eps)).alias("trend_gmv_30d_90d"),
        (pl.col("gmv_sum_14d") / (pl.col("gmv_sum_30d") + eps)).alias("trend_gmv_14d_30d"),
        (pl.col("gmv_sum_30d") / (pl.col("gmv_sum_180d") + eps)).alias("trend_gmv_30d_180d"),
        (pl.col("gmv_sum_90d") / (pl.col("gmv_sum_365d") + eps)).alias("trend_gmv_90d_365d"),
        (pl.col("to_ord_sum_7d") / (pl.col("to_ord_sum_30d") + eps)).alias("trend_ord_7d_30d"),
    ])
    
    exprs.extend([
        (pl.col("search_to_cart_sum_30d") / (pl.col("searches_sum_30d") + eps)).alias("cr_search_to_cart_30d"),
        (pl.col("to_ord_sum_30d") / (pl.col("to_cart_sum_30d") + eps)).alias("cr_cart_to_ord_30d"),
        (pl.col("search_to_ord_sum_30d") / (pl.col("searches_sum_30d") + eps)).alias("cr_search_to_ord_30d"),
        (pl.col("cat_to_cart_sum_30d") / (pl.col("cat_sum_30d") + eps)).alias("cr_cat_to_cart_30d"),
        (pl.col("cat_to_ord_sum_30d") / (pl.col("cat_sum_30d") + eps)).alias("cr_cat_to_ord_30d"),
    ])
    
    exprs.extend([
        (pl.col("gmv_sum_30d") / (pl.col("to_ord_sum_30d") + eps)).alias("aov_30d"),
        (pl.col("gmv_sum_90d") / (pl.col("to_ord_sum_90d") + eps)).alias("aov_90d"),
        (pl.col("gmv_sum_180d") / (pl.col("to_ord_sum_180d") + eps)).alias("aov_180d"),
        (pl.col("gmv_sum_30d") / (pl.col("active_days_30d") + eps)).alias("gmv_per_active_day_30d"),
        (pl.col("gmv_sum_90d") / (pl.col("active_days_90d") + eps)).alias("gmv_per_active_day_90d"),
        (pl.col("gmv_cat_sum_30d") / (pl.col("gmv_search_sum_30d") + pl.col("gmv_cat_sum_30d") + eps)).alias("ratio_gmv_cat_30d"),
    ])
    
    exprs.extend([
        (pl.col("std_order_gap") / (pl.col("mean_order_gap").abs() + eps)).alias("order_regularity_cv"),
        (pl.col("std_cart_gap") / (pl.col("mean_cart_gap").abs() + eps)).alias("cart_regularity_cv"),
        (pl.col("std_active_gap") / (pl.col("mean_active_gap").abs() + eps)).alias("activity_regularity_cv"),
        (pl.col("recency_ord_days").cast(pl.Float64) / (pl.col("mean_order_gap").abs() + eps)).alias("order_overdue_ratio"),
        (pl.lit(30.0) / (pl.col("mean_order_gap").abs() + eps)).alias("expected_orders_30d"),
        (pl.col("n_order_days").cast(pl.Float64) / (pl.col("n_active_days_total").cast(pl.Float64) + eps)).alias("order_intensity"),
        (pl.col("std_gmv_per_order_day") / (pl.col("avg_gmv_per_order_day") + eps)).alias("gmv_consistency"),
    ])
    
    exprs.extend([
        ((pl.col("gmv_period_w1") + pl.col("gmv_period_w2")) / 
         (pl.col("gmv_period_w3") + pl.col("gmv_period_w4") + eps)).alias("gmv_trajectory"),
        ((pl.col("active_period_w1") + pl.col("active_period_w2")).cast(pl.Float64) /
         (pl.col("active_period_w3").cast(pl.Float64) + pl.col("active_period_w4").cast(pl.Float64) + eps)).alias("activity_trajectory"),
        ((pl.col("ord_period_w1") + eps) / (pl.col("ord_period_w2") + eps) -
         (pl.col("ord_period_w3") + eps) / (pl.col("ord_period_w4") + eps)).alias("order_acceleration"),
        ((pl.col("gmv_period_w1") + pl.col("gmv_period_w2")) > 0).cast(pl.Float64).alias("is_recent_buyer"),
        ((pl.col("active_period_w1") + pl.col("active_period_w2")) > 0).cast(pl.Float64).alias("is_recently_active"),
    ])
    
    exprs.extend([
        (pl.col("search_share_30d") - pl.col("search_share_90d")).alias("channel_shift_30_90"),
        (pl.col("search_share_30d") - pl.col("search_share_180d")).alias("channel_shift_30_180"),
        (pl.col("recency_ord_days") > 120).cast(pl.Float64).alias("is_dormant"),
    ])
    
    return exprs

In [14]:
def generate_features_and_targets(
    data: pl.DataFrame,
    anchor: date,
    user_batch: list[int],
    is_train: bool = True
) -> pl.DataFrame:
    max_back = max(w[1] for w in WINDOWS)
    data_f = data.filter(
        pl.col("user_id").is_in(user_batch) &
        (pl.col("event_date") <= anchor) &
        (pl.col("event_date") >= anchor - timedelta(days=max_back))
    )

    if len(data_f) > 0:
        all_exprs = (
            _build_feature_exprs(anchor, WINDOWS, VALUE_COLS) +
            _build_interpurchase_exprs(anchor) +
            _build_lifecycle_exprs(anchor)
        )
        features = (
            data_f.group_by("user_id")
            .agg(all_exprs)
            .with_columns(_build_secondary_exprs())
            .with_columns(anchor_date=pl.lit(anchor))
        )
    else:
        features = pl.DataFrame({"user_id": user_batch, "anchor_date": anchor})

    index_df = pl.DataFrame({"user_id": user_batch}).with_columns(anchor_date=pl.lit(anchor))
    result = index_df.join(features, on=["anchor_date", "user_id"], how="left")

    skip_fill = ["anchor_date", "user_id", "recency_ord_days", "recency_search_days",
                 "recency_cart_days", "recency_cat_days"]
    feat_cols = [c for c in result.columns if c not in skip_fill]
    result = result.with_columns([pl.col(c).fill_null(0.0) for c in feat_cols])

    if is_train:
        t_start = anchor + timedelta(days=1)
        t_end   = anchor + timedelta(days=30)
        targets = (
            data.filter(
                pl.col("user_id").is_in(user_batch) &
                pl.col("event_date").is_between(t_start, t_end)
            )
            .group_by("user_id")
            .agg(pl.col(TARGET_COL).sum().clip(upper_bound=4000.0).alias("target"))
        )
        result = result.join(targets, on="user_id", how="left").with_columns(pl.col("target").fill_null(0.0))
    else:
        result = result.with_columns(pl.lit(None).cast(pl.Float64).alias("target"))

    return result

In [15]:
print("Читаем данные...")
data = pl.read_parquet(DATA_DIR / 'train.parquet')
user_ids = data["user_id"].unique().sort().to_list()
n_batches = (len(user_ids) + BATCH_SIZE - 1) // BATCH_SIZE

anchors = generate_cv_anchor_dates(data, n_folds=N_FOLDS)
anchor_test = data["event_date"].max()

print(f"Пользователей: {len(user_ids):,}")
print(f"Период: {data['event_date'].min()} — {data['event_date'].max()}")
print(f"Фолды: {len(anchors)} ({anchors[0]} → {anchors[-1]})")
print(f"Тест: {anchor_test}")
print(f"Батчей: {n_batches}")

t0 = time.time()
for fold_idx, anchor in enumerate(anchors):
    fold_dir = FEATURES_DIR / f"fold_{fold_idx:02d}"
    fold_dir.mkdir(parents=True, exist_ok=True)
    for batch in range(n_batches):
        current_batch = user_ids[batch * BATCH_SIZE : (batch + 1) * BATCH_SIZE]
        out_df = generate_features_and_targets(data, anchor, current_batch, is_train=True)
        out_df.write_parquet(fold_dir / f"batch_{batch:04d}.parquet")
    print(f"  fold_{fold_idx:02d} ({anchor}) — {time.time()-t0:.0f}s")

fold_dir = FEATURES_DIR / "fold_test"
fold_dir.mkdir(parents=True, exist_ok=True)
for batch in range(n_batches):
    current_batch = user_ids[batch * BATCH_SIZE : (batch + 1) * BATCH_SIZE]
    out_df = generate_features_and_targets(data, anchor_test, current_batch, is_train=False)
    out_df.write_parquet(fold_dir / f"batch_{batch:04d}.parquet")
print(f"  fold_test ({anchor_test}) — {time.time()-t0:.0f}s")

test_check = pl.read_parquet(FEATURES_DIR / "fold_test" / "batch_*.parquet")
drop_cols = ["user_id", "anchor_date", "target"]
features_list = [c for c in test_check.columns if c not in drop_cols]
print(f"\nИтого фичей: {len(features_list)}")
print(f"Размер fold_test: {test_check.shape}")
print(f"Время: {time.time()-t0:.0f}s")

new_feats = [f for f in features_list if any(k in f for k in [
    'gap', 'regularity', 'overdue', 'expected_orders', 'intensity', 'consistency',
    'n_order', 'n_cart_days', 'n_active_days_total', 'avg_gmv_per_order', 'std_gmv_per_order',
    'trajectory', 'acceleration', 'period_w', 'is_recent', 'channel_shift', 'search_share'
])]
print(f"\nНовых фич v6: {len(new_feats)}")
for f in sorted(new_feats):
    print(f"  {f}")

Читаем данные...
Пользователей: 250,000
Период: 2025-01-01 — 2026-02-13
Фолды: 5 (2025-11-19 → 2026-01-14)
Тест: 2026-02-13
Батчей: 5
  fold_00 (2025-11-19) — 42s
  fold_01 (2025-12-03) — 81s
  fold_02 (2025-12-17) — 116s
  fold_03 (2025-12-31) — 155s
  fold_04 (2026-01-14) — 192s
  fold_test (2026-02-13) — 228s

Итого фичей: 437
Размер fold_test: (250000, 440)
Время: 229s

Новых фич v6: 43
  active_period_w1
  active_period_w2
  active_period_w3
  active_period_w4
  activity_regularity_cv
  activity_trajectory
  avg_gmv_per_order_day
  cart_regularity_cv
  channel_shift_30_180
  channel_shift_30_90
  expected_orders_30d
  gmv_consistency
  gmv_period_w1
  gmv_period_w2
  gmv_period_w3
  gmv_period_w4
  gmv_trajectory
  is_recent_buyer
  is_recently_active
  max_order_gap
  mean_active_gap
  mean_cart_gap
  mean_order_gap
  median_order_gap
  min_order_gap
  n_active_days_total
  n_cart_days
  n_order_days
  ord_period_w1
  ord_period_w2
  ord_period_w3
  ord_period_w4
  order_accelera